# LeetCode #1244: Design A Leaderboard

https://leetcode.com/problems/design-a-leaderboard/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Sorted List)** | $O(n)$ addScore, $O(n \log n)$ top | $O(n)$ |
| **Optimal: HashMap + Sort-on-Query ★** | $O(1)$ addScore, $O(n \log n)$ top, $O(1)$ reset | $O(n)$ |

---

## Understanding the Methods

### Brute Force (Sorted List)
Maintain a sorted list of (score, player) pairs. addScore requires finding and reinserting the player ($O(n)$); top sums the first $k$ elements ($O(k)$).

### Optimal: HashMap + Sort-on-Query ★
Store player scores in a hash map for $O(1)$ add and reset. For `top(k)`, sort the values descending and sum the first $k$ — this is $O(n \log n)$ but called infrequently and requires no extra data structure per mutation.

**Why this is better than Brute Force:** Mutations (the hot path) are $O(1)$; the sort-on-query pattern is simple and correct without maintaining a complex sorted structure.

**Constraints:**
* $1 \le playerId, score \le 10^6$
* $1 \le K \le$ current number of players in the leaderboard
* At most $10^4$ calls to each function

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
using System.Linq;
public class Leaderboard {
    // Maps each player to their accumulated score for O(1) mutation
    private Dictionary<int, int> scores = new();
    public void AddScore(int playerId, int score) {
        scores.TryGetValue(playerId, out int cur);
        scores[playerId] = cur + score;
    }
    public int Top(int K) {
        // Sort all scores descending and sum the top K entries
        return scores.Values.OrderByDescending(s => s).Take(K).Sum();
    }
    public void Reset(int playerId) {
        // Remove the player entirely — their score resets to 0
        scores.Remove(playerId);
    }
}

### Python

In [ ]:
class Leaderboard:
    def __init__(self):
        # Maps each player to their accumulated score for O(1) mutation
        self.scores: dict[int, int] = {}

    def add_score(self, player_id: int, score: int) -> None:
        self.scores[player_id] = self.scores.get(player_id, 0) + score

    def top(self, k: int) -> int:
        # Sort all scores descending and sum the top k entries
        return sum(sorted(self.scores.values(), reverse=True)[:k])

    def reset(self, player_id: int) -> None:
        # Remove the player entirely — their score resets to 0
        del self.scores[player_id]

### Go

In [ ]:
import "sort"
type Leaderboard struct {
    // Maps each player to their accumulated score for O(1) mutation
    scores map[int]int
}
func Constructor() Leaderboard { return Leaderboard{map[int]int{}} }
func (l *Leaderboard) AddScore(playerId int, score int) {
    l.scores[playerId] += score
}
func (l *Leaderboard) Top(k int) int {
    vals := make([]int, 0, len(l.scores))
    for _, v := range l.scores { vals = append(vals, v) }
    // Sort all scores descending and sum the top k entries
    sort.Sort(sort.Reverse(sort.IntSlice(vals)))
    sum := 0
    for i := 0; i < k; i++ { sum += vals[i] }
    return sum
}
func (l *Leaderboard) Reset(playerId int) {
    // Remove the player entirely — their score resets to 0
    delete(l.scores, playerId)
}

### Rust

In [ ]:
use std::collections::HashMap;
struct Leaderboard {
    // Maps each player to their accumulated score for O(1) mutation
    scores: HashMap<i32, i32>,
}
impl Leaderboard {
    fn new() -> Self { Leaderboard { scores: HashMap::new() } }
    fn add_score(&mut self, player_id: i32, score: i32) {
        *self.scores.entry(player_id).or_insert(0) += score;
    }
    fn top(&self, k: i32) -> i32 {
        // Sort all scores descending and sum the top k entries
        let mut vals: Vec<i32> = self.scores.values().cloned().collect();
        vals.sort_unstable_by(|a, b| b.cmp(a));
        vals.iter().take(k as usize).sum()
    }
    fn reset(&mut self, player_id: i32) {
        // Remove the player entirely — their score resets to 0
        self.scores.remove(&player_id);
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `addScore(1,73)`, `addScore(2,56)`, `addScore(3,39)`, `top(1)` → **73**, `reset(1)`, `top(1)` → **56**
After three adds, the map is `{1:73, 2:56, 3:39}`; top-1 is 73. After resetting player 1, top-1 is 56.

### 2. Slightly Complex
**Input:** `addScore(1,100)`, `addScore(1,50)` → player 1 now has 150. `top(2)` with only 1 player returns 150.
Hash map accumulation handles repeated adds correctly: `scores[1] = 100 + 50 = 150`.

### 3. Edge Case: Time Factor
**Input:** $10^4$ calls to `top(k)` with $n = 10^4$ players.
Each `top` sorts $10^4$ values ($\approx 10^4 \log 10^4 \approx 130{,}000$ comparisons). $10^4$ such calls total $\approx 1.3 \times 10^9$ operations — tight but acceptable given the problem's constraints.

### 4. Edge Case: Space Factor
**Input:** $10^4$ distinct players each with one `addScore`.
The hash map holds $10^4$ entries — $O(n)$ space — which is the unavoidable lower bound.

### 5. Almost-Impossible but Plausible
**Input:** `addScore(1, 10^6)` repeated $10^4$ times, then `top(1)`.
Player 1's score reaches $10^{10}$; C# `int` (32-bit) overflows. The sum in `Top` should use `long` for production — the example exposes this subtle overflow risk.